In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')

In [4]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [6]:
X_train

,age,gender,fever,cough,city
30,15,Male,101.0,Mild,Delhi
79,48,Female,103.0,Mild,Kolkata
0,60,Male,103.0,Mild,Kolkata
97,20,Female,101.0,Mild,Bangalore
3,31,Female,98.0,Mild,Kolkata
...,...,...,...,...,...
31,83,Male,103.0,Mild,Kolkata
16,69,Female,103.0,Mild,Kolkata
77,8,Female,101.0,Mild,Kolkata
94,79,Male,NaN,Strong,Kolkata


# Without using column transformer

In [7]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [16]:
#adding simple imputer to fever column -> fills the missing value with mean
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [17]:
df['cough'].unique()

array(['Mild', 'Strong'], dtype=object)

In [18]:
# Ordinal Encoding -> Cough column
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

#also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [26]:
df['city'].unique()

array(['Kolkata', 'Delhi', 'Mumbai', 'Bangalore'], dtype=object)

In [19]:
# Onehotencoding in gender, city column

ohe = OneHotEncoder(drop = 'first', sparse_output= False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

#for the test data 
X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])

In [20]:
X_test_gender_city.shape

(20, 4)

In [21]:
X_train_gender_city.shape

(80, 4)

## Now concatinating

In [22]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

# also the test data 
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_train_age.shape, X_test_age.shape


((80, 1), (20, 1))

In [23]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis = 1)
#for test data 
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis = 1)

In [25]:
X_train_transformed.shape

(80, 7)

## This is hectic and tiresome and not efficient.

# **Using ColumnTransformer**

In [28]:
from sklearn.compose import ColumnTransformer


In [30]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']), 
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']), 
    ('tnf3', OneHotEncoder(sparse_output=False, drop= 'first'),['gender', 'city'] )
], remainder='passthrough')
#remainder parameter is for what to do with the remaining columns; 'drop' or 'passthrough'

In [34]:
# transformer.fit_transform
transformer.fit_transform(X_train).shape

(80, 7)

In [35]:
transformer.transform(X_test).shape

(20, 7)